We begin by importing our dataset by directly downloading the file from Kaggle with the help of the following statement.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jessicali9530/caltech256")

print("Path to dataset files:", path)

100%|██████████| 2.12G/2.12G [00:12<00:00, 177MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jessicali9530/caltech256/versions/2


In [2]:
import os
from torchvision import datasets, transforms


dataset_dir = os.path.join(path, "256_ObjectCategories")
transform = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor()])
dataset = datasets.ImageFolder(
    root=dataset_dir,
    transform=transform
)

print("Number of images:", len(dataset))
print("Number of classes:", len(dataset.classes))

Number of images: 30607
Number of classes: 257


In [3]:
x, y = dataset[0]

print(x.shape)  # torch.Size([3, 224, 224])
print(y)        # class index

torch.Size([3, 224, 224])
0


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets


class CustomCaltechCNN(nn.Module):
    def __init__(self, num_classes=257):
        super(CustomCaltechCNN, self).__init__()

        #Take input (3, 224, 224) and obtain after pooling: (32, 112, 112)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        #Take previous input (32, 112, 112) and obtain after pooling: (64, 56, 56)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        #Take previous input (64, 56, 56) and obtain after pooling: (128, 28, 28)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(
        in_channels=128,
        out_channels=256,
        kernel_size=3,
        stride=1,
        padding=1)

        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.global_pool = nn.AdaptiveAvgPool2d((1
        , 1))

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
      x = self.pool(F.relu(self.bn1(self.conv1(x))))
      x = self.pool(F.relu(self.bn2(self.conv2(x))))
      x = self.pool(F.relu(self.bn3(self.conv3(x))))
      x = self.pool(F.relu(self.bn4(self.conv4(x))))

      x = self.global_pool(x)

      x = torch.flatten(x, 1)

      x = self.dropout(F.relu(self.fc1(x)))
      x = self.fc2(x)
      return x


We have defined our function. Now, we will call and train it in the following cell.

Now, we will import, prepare data and train the model.

In [ ]:
if __name__ == '__main__':
    #we have already downloaded the dataset and using the previously called directory, we use it here
    data_directory = dataset_dir
    batch_sizee = 32
    epochs = 20
    learning_rate = 0.001
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing pipeline on device accelerator: {device}")

    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])])
    #Why these specific numbers?
    #mean=[0.485, 0.456, 0.406]
    #std=[0.229, 0.224, 0.225]
    #These are the channel-wise mean and standard deviation of the ImageNet dataset.




    #Now, we import our data and split into train and test sets
    full_dataset = datasets.ImageFolder(root=data_directory, transform=data_transforms)

    num_classes = len(full_dataset.classes)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_sizee, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_sizee, shuffle=False, num_workers=2)

    #Initializing the model by calling the previously defined function
    model = CustomCaltechCNN(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    for epoch in range(epochs):
      model.train()
      running_loss = 0.0
      correct = 0
      total = 0

      for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

      train_accuracy = 100 * correct / total

      print(
          f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Train Acc: {train_accuracy:.2f}% "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}"
      )
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Final Test Partition Accuracy: {accuracy:.2f}%\n")

Executing pipeline on device accelerator: cuda
Epoch [1/20] Loss: 5.1655 Train Acc: 6.43% LR: 0.001000
Epoch [2/20] Loss: 4.8369 Train Acc: 9.12% LR: 0.001000
Epoch [3/20] Loss: 4.6342 Train Acc: 10.95% LR: 0.001000
Epoch [4/20] Loss: 4.4686 Train Acc: 12.60% LR: 0.001000
Epoch [5/20] Loss: 4.3020 Train Acc: 14.52% LR: 0.001000
Epoch [6/20] Loss: 4.1621 Train Acc: 16.50% LR: 0.001000
Epoch [7/20] Loss: 4.0423 Train Acc: 17.45% LR: 0.001000
Epoch [8/20] Loss: 3.9433 Train Acc: 18.67% LR: 0.001000
Epoch [9/20] Loss: 3.8623 Train Acc: 19.48% LR: 0.001000
Epoch [10/20] Loss: 3.7751 Train Acc: 20.84% LR: 0.001000
Epoch [11/20] Loss: 3.7067 Train Acc: 21.73% LR: 0.001000
Epoch [12/20] Loss: 3.6411 Train Acc: 22.45% LR: 0.001000
Epoch [13/20] Loss: 3.5689 Train Acc: 23.69% LR: 0.001000
Epoch [14/20] Loss: 3.5172 Train Acc: 24.22% LR: 0.001000
Epoch [15/20] Loss: 3.4484 Train Acc: 25.31% LR: 0.001000
Epoch [16/20] Loss: 3.4012 Train Acc: 26.10% LR: 0.001000
Epoch [17/20] Loss: 3.3447 Train Acc

We see that the loss decreased and training accuracy is increasing continuously, so we will choose to continue training the model for further epochs.

In [ ]:
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "caltech_checkpoint.pth")
model = CustomCaltechCNN(num_classes=num_classes).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [12]:
checkpoint = torch.load(
    "caltech_checkpoint.pth",
    map_location=device
)

model.load_state_dict(
    checkpoint['model_state_dict']
)

optimizer.load_state_dict(
    checkpoint['optimizer_state_dict']
)

start_epoch = checkpoint['epoch']
additional_epochs = 30





In [13]:
for epoch in range(start_epoch,start_epoch + additional_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()
          running_loss += loss.item()
          _, predicted = torch.max(outputs, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{start_epoch+additional_epochs}] "
        f"Loss: {running_loss/len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Final Test Partition Accuracy: {accuracy:.2f}%\n")

Final Test Partition Accuracy: 38.58%



In [15]:
print(
        f"Epoch [{epoch+1}/{start_epoch+additional_epochs}] "
        f"Loss: {running_loss/len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
    )

Epoch [50/50] Loss: 2.1723 Train Acc: 46.78% 


In [16]:
accuracy = 100 * correct / total
print(f"Final Test Partition Accuracy: {accuracy:.2f}%\n")

Final Test Partition Accuracy: 38.58%



In [20]:
model.eval()

correcte = 0
totale = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        # predicted class index
        _, predicted = torch.max(outputs, dim=1)

        totale += labels.size(0)
        correcte += (predicted == labels).sum().item()

test_accuracy = 100 * correcte / totale
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 38.58%


In [21]:
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "caltech_checkpoint.pth")

In [22]:
from google.colab import files

files.download("caltech_checkpoint.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
checkpoint = torch.load("caltech_checkpoint.pth", map_location="cpu")

print(checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict'])
